# Clustering Feature Engineering

This notebook builds customer-level features from the cleaned transaction files and merges them into the clustering dataset.

In [10]:
# Load master_customer_dataframe.csv, remove specified columns, and save as Clustering.csv
import pandas as pd
from pathlib import Path

# Load master_customer_dataframe
master_df_path = Path(r"E:\After 1-16 Cleaned\Cleaned_Original\feature_engineering_ipynb\master_customer_dataframe.csv")
master_df = pd.read_csv(master_df_path)

print("Loaded master_customer_dataframe.csv")
print(f"Shape: {master_df.shape}")
print(f"Columns: {master_df.columns.tolist()}")

# Remove specified columns
cols_to_remove = ['country', 'sales_or_income', 'onboard_date', 'establish_date', 'birth_establish_date']
cols_to_remove = [col for col in cols_to_remove if col in master_df.columns]
if cols_to_remove:
    master_df = master_df.drop(columns=cols_to_remove)
    print(f"\nRemoved columns: {cols_to_remove}")
else:
    print("\nNo specified columns found to remove")

print(f"\nShape after removal: {master_df.shape}")
print(f"Columns after removal: {master_df.columns.tolist()}")

# Save as Clustering.csv
clustering_output_path = Path(r"E:\After 1-16 Cleaned\Cleaned_Original\Clustering\Clustering.csv")
master_df.to_csv(clustering_output_path, index=False)
print(f"\nSaved to: {clustering_output_path}")
print(f"Total rows: {len(master_df)}")
print(f"Total columns: {len(master_df.columns)}")

Loaded master_customer_dataframe.csv
Shape: (61410, 14)
Columns: ['customer_id', 'kyc_type', 'country', 'province', 'city', 'industry_code', 'occupation_code', 'sales_or_income', 'income_cents', 'sales_cents', 'account_age', 'onboard_date', 'birth_establish_date', 'birth_business_age']

Removed columns: ['country', 'sales_or_income', 'onboard_date', 'birth_establish_date']

Shape after removal: (61410, 10)
Columns after removal: ['customer_id', 'kyc_type', 'province', 'city', 'industry_code', 'occupation_code', 'income_cents', 'sales_cents', 'account_age', 'birth_business_age']

Saved to: E:\After 1-16 Cleaned\Cleaned_Original\Clustering\Clustering.csv
Total rows: 61410
Total columns: 10


In [11]:
from pathlib import Path
import numpy as np
import pandas as pd

clustering = pd.read_csv('../Clustering/Clustering.csv')
individual = pd.read_csv('individual_cleaned_Jan16.csv')
business = pd.read_csv('business_cleaned_Jan16.csv')

transaction_files = {
    "ABM": 'abm_cleaned.csv',
    "Card": 'Card_Cleaned.csv',
    "Cheque": 'Cleaned_cheque.csv',
    "EFT": 'eft.csv',
    "EMT": 'emt.csv',
    "Western Union": 'westernunion.csv',
    "Wire": 'Cleaned_wire.csv',
}

clustering.head()

,customer_id,kyc_type,province,city,industry_code,occupation_code,income_cents,sales_cents,account_age,birth_business_age
0,SYNID0108229423,individual,other,other,NaN,13100,5185000.0,NaN,29.62,68.65
1,SYNID0102628368,individual,ON,WINDSOR,NaN,41101,7227000.0,NaN,8.87,57.61
2,SYNID0107013748,individual,BC,SURREY,NaN,42202,-1.0,NaN,3.83,26.42
3,SYNID0109706496,individual,ON,BRAMPTON,NaN,10019,1000.0,NaN,8.02,58.92
4,SYNID0100925078,individual,ON,TORONTO,NaN,OTHER,7372900.0,NaN,32.49,66.61


In [12]:
def normalize_debit_credit(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.upper().replace({"DEBIT": "D", "CREDIT": "C"})


def normalize_bool(series: pd.Series) -> pd.Series:
    normalized = series.astype(str).str.strip().str.upper()
    return normalized.isin(["1", "TRUE", "T", "YES", "Y"])


# Load and normalize transaction files
transaction_frames = []
required_geo_columns = ["country", "province", "city"]

for txn_type, path in transaction_files.items():
    df = pd.read_csv(path)
    df["transaction_type"] = txn_type
    df["debit_credit"] = normalize_debit_credit(df["debit_credit"])
    df["amount_cad"] = pd.to_numeric(df["amount_cad"], errors="coerce")

    for col in required_geo_columns:
        if col not in df.columns:
            df[col] = np.nan

    transaction_frames.append(df)

transactions = pd.concat(transaction_frames, ignore_index=True)

# Overall transaction counts and by type
transaction_counts = transactions.groupby("customer_id").size().rename("total_transaction_count")
transaction_counts_by_type = (
    transactions.groupby(["customer_id", "transaction_type"]).size().unstack(fill_value=0)
)
transaction_counts_by_type = transaction_counts_by_type.add_prefix("transaction_count_")

# Debit/Credit counts and sums
is_debit = transactions["debit_credit"] == "D"
is_credit = transactions["debit_credit"] == "C"


debit_stats = (
    transactions[is_debit]
    .groupby("customer_id")
    .agg(debit_transaction_count=("transaction_id", "size"), debit_transaction_amount=("amount_cad", "sum"))
)
credit_stats = (
    transactions[is_credit]
    .groupby("customer_id")
    .agg(credit_transaction_count=("transaction_id", "size"), credit_transaction_amount=("amount_cad", "sum"))
)

# Average transaction amount split by debit and credit
avg_amount_debit = (
    transactions[is_debit]
    .groupby("customer_id")["amount_cad"]
    .mean()
    .rename("average_debit_transaction_amount")
)
avg_amount_credit = (
    transactions[is_credit]
    .groupby("customer_id")["amount_cad"]
    .mean()
    .rename("average_credit_transaction_amount")
)

# Geographic features
geo_stats = (
    transactions.groupby("customer_id")
    .agg(
        unique_transaction_cities=("city", pd.Series.nunique),
        unique_transaction_provinces=("province", pd.Series.nunique),
        unique_transaction_countries=("country", pd.Series.nunique),
    )
)
geo_stats["multi_country_transactions"] = (geo_stats["unique_transaction_countries"] > 1).astype(int)
geo_stats = geo_stats.drop(columns=["unique_transaction_countries"])

# Card-specific features
card = pd.read_csv(transaction_files["Card"])
card["merchant_category"] = card["merchant_category"].astype(str).str.strip()

card_total = card.groupby("customer_id")["transaction_id"].size().rename("card_transaction_count")
card_ecom = normalize_bool(card["ecommerce_ind"]).groupby(card["customer_id"]).sum().rename("card_ecommerce_count")
card_non_ecom = (card_total - card_ecom).rename("card_non_ecommerce_count")

card_ratio = (card_ecom / card_non_ecom.replace(0, np.nan)).rename("card_ecommerce_to_nonecommerce_ratio")
card_mcc_diversity = card.groupby("customer_id")["merchant_category"].nunique().rename("card_mcc_diversity_count")

card_mcc_counts = (
    card.groupby(["customer_id", "merchant_category"]).size().rename("mcc_count").reset_index()
)
card_mcc_counts_sorted = card_mcc_counts.sort_values(["customer_id", "mcc_count", "merchant_category"], ascending=[True, False, True])
card_top2 = card_mcc_counts_sorted.groupby("customer_id").head(2)
card_top2_pivot = (
    card_top2.assign(rank=card_top2.groupby("customer_id").cumcount() + 1)
    .pivot(index="customer_id", columns="rank", values="merchant_category")
    .rename(columns={1: "card_top_mcc", 2: "card_second_mcc"})
)
card_mcc_weight = (
    card_mcc_counts_sorted.groupby("customer_id")["mcc_count"].max() / card_total
).rename("card_top_mcc_weight")

card_features = pd.concat(
    [card_total, card_ecom, card_non_ecom, card_ratio, card_mcc_diversity, card_mcc_weight, card_top2_pivot],
    axis=1,
)

# ABM-specific features
abm = pd.read_csv(transaction_files["ABM"])
abm_total = abm.groupby("customer_id")["transaction_id"].size().rename("abm_transaction_count")
abm_cash = normalize_bool(abm["cash_indicator"]).groupby(abm["customer_id"]).sum().rename("abm_cash_count")
abm_non_cash = (abm_total - abm_cash).rename("abm_non_cash_count")
abm_cash_ratio = (abm_cash / abm_non_cash.replace(0, np.nan)).rename("abm_cash_to_non_cash_ratio")

abm_features = pd.concat([abm_total, abm_cash, abm_non_cash, abm_cash_ratio], axis=1)

# Calculate Debit_Amount_Total and Credit_Amount_Total (total amounts across all transaction channels)
Debit_Amount_Total = (
    transactions[is_debit]
    .groupby("customer_id")["amount_cad"]
    .sum()
    .rename("Debit_Amount_Total")
)

Credit_Amount_Total = (
    transactions[is_credit]
    .groupby("customer_id")["amount_cad"]
    .sum()
    .rename("Credit_Amount_Total")
)

# Calculate Averagetime_between_alltransaction (average time between transactions for each customer)
# Convert transaction_datetime to datetime
transactions["transaction_datetime"] = pd.to_datetime(transactions["transaction_datetime"], errors="coerce")

# Sort by customer_id and transaction_datetime
transactions_sorted = transactions.sort_values(["customer_id", "transaction_datetime"]).copy()

# Calculate time differences between consecutive transactions for each customer
transactions_sorted["time_diff"] = (
    transactions_sorted.groupby("customer_id")["transaction_datetime"].diff()
)

# Calculate average time between transactions for each customer (in days)
Averagetime_between_alltransaction = (
    transactions_sorted.groupby("customer_id")["time_diff"]
    .mean()
    .dt.total_seconds()
    .divide(86400)  # Convert to days
    .rename("Averagetime_between_alltransaction")
)

# Calculate standard deviation of time between transactions (in days)
std_time_between_transactions = (
    transactions_sorted.groupby("customer_id")["time_diff"]
    .std()
    .dt.total_seconds()
    .divide(86400)  # Convert to days
    .rename("std_time_between_transactions")
)

# Calculate median time between transactions (in days)
median_time_between_transactions = (
    transactions_sorted.groupby("customer_id")["time_diff"]
    .median()
    .dt.total_seconds()
    .divide(86400)  # Convert to days
    .rename("median_time_between_transactions")
)

# Calculate % of transactions in top 10% busiest days
# Extract date from transaction_datetime
transactions_sorted["transaction_date"] = transactions_sorted["transaction_datetime"].dt.date

# Count transactions per day for each customer
daily_transaction_counts = (
    transactions_sorted.groupby(["customer_id", "transaction_date"])
    .size()
    .reset_index(name="daily_count")
)

# For each customer, find top 10% busiest days
def calculate_top10_pct_transactions(group):
    if len(group) == 0:
        return np.nan
    # Calculate how many days represent top 10%
    n_days = len(group)
    top10_pct_days = max(1, int(np.ceil(n_days * 0.1)))  # At least 1 day
    # Get top 10% busiest days
    top_days = group.nlargest(top10_pct_days, "daily_count")
    # Calculate total transactions on those days
    total_on_top_days = top_days["daily_count"].sum()
    # Calculate percentage
    total_transactions = group["daily_count"].sum()
    return (total_on_top_days / total_transactions * 100) if total_transactions > 0 else np.nan

pct_transactions_top10_busiest_days = (
    daily_transaction_counts.groupby("customer_id")
    .apply(calculate_top10_pct_transactions)
    .rename("pct_transactions_top10_busiest_days")
)

# Combine all features
features = pd.concat(
    [
        transaction_counts,
        transaction_counts_by_type,
        debit_stats,
        credit_stats,
        avg_amount_debit,
        avg_amount_credit,
        geo_stats,
        card_features,
        abm_features,
        Debit_Amount_Total,
        Credit_Amount_Total,
        Averagetime_between_alltransaction,
        std_time_between_transactions,
        median_time_between_transactions,
        pct_transactions_top10_busiest_days,
    ],
    axis=1,
).reset_index()

clustering_enriched = clustering.merge(features, on="customer_id", how="left")

if "income_cents" in clustering_enriched.columns:
    clustering_enriched = clustering_enriched.drop(columns=["income_cents"])

clustering_enriched.head()

C:\Users\houju\AppData\Local\Temp\ipykernel_47868\1483166485.py:198: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calculate_top10_pct_transactions)


,customer_id,kyc_type,province,city,industry_code,occupation_code,sales_cents,account_age,birth_business_age,total_transaction_count,...,abm_transaction_count,abm_cash_count,abm_non_cash_count,abm_cash_to_non_cash_ratio,Debit_Amount_Total,Credit_Amount_Total,Averagetime_between_alltransaction,std_time_between_transactions,median_time_between_transactions,pct_transactions_top10_busiest_days
0,SYNID0108229423,individual,other,other,NaN,13100,NaN,29.62,68.65,114,...,2.0,1.0,1.0,1.0,803387.0,12299.0,0.787278,0.712698,0.607888,20.175439
1,SYNID0102628368,individual,ON,WINDSOR,NaN,41101,NaN,8.87,57.61,8,...,NaN,NaN,NaN,NaN,146360.0,228053.0,12.878022,8.829264,12.000000,12.500000
2,SYNID0107013748,individual,BC,SURREY,NaN,42202,NaN,3.83,26.42,149,...,7.0,6.0,1.0,6.0,6381165.4,2539715.0,0.617069,0.709136,0.377025,20.805369
3,SYNID0109706496,individual,ON,BRAMPTON,NaN,10019,NaN,8.02,58.92,3,...,NaN,NaN,NaN,NaN,108576.0,3359.0,12.672078,13.335715,3.242303,33.333333
4,SYNID0100925078,individual,ON,TORONTO,NaN,OTHER,NaN,32.49,66.61,39,...,NaN,NaN,NaN,NaN,1320470.0,1677447.0,2.357771,2.720995,1.350775,23.076923


In [13]:
output_path = Path(r"E:\After 1-16 Cleaned\Cleaned_Original\Clustering\clustering_Cleaned.csv")
clustering_enriched.to_csv(output_path, index=False)
output_path

WindowsPath('E:/After 1-16 Cleaned/Cleaned_Original/Clustering/clustering_Cleaned.csv')